# Session 1 — Orientation: from LLM calls to agentic systems

**Goal:** see the raw request/response boundary with no framework, and write down what *reliable enough* means for this course.

Everything runs on the deterministic `FakeLLM`. `LIVE` is your configured lane (ollama by default in class); when it is down the preflight says so and `LIVE` is a `FakeLLM` too.

In [ ]:
# Preflight: environment checks with a fix for anything missing. It never raises.
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))
CORPUS_DIR = REPO_ROOT / "data" / "corpus"

try:
    from bootcamp_agent.preflight import preflight
except ImportError:
    print("❌ bootcamp_agent not importable -> in the repo root run: uv sync --group dev")
    print("   then pick the .venv kernel (or start Jupyter with: uv run jupyter lab)")
else:
    LIVE = preflight(REPO_ROOT)  # the configured lane's client; FakeLLM whenever the lane is down

In [ ]:
from bootcamp_agent.checks import check, review

## 1. The smallest possible 'agent'

One call: a system instruction, a user question, a text answer. This is the boundary every framework wraps. The `FakeLLM` answers from a keyword table, so the same question always gets the same answer.

In [ ]:
from bootcamp_agent.llm import FakeLLM

hello_llm = FakeLLM(
    responses={
        "hello": "Hello! I am a deterministic stand-in for a language model.",
        "agent": "An agent is a loop around a model: perceive, decide, act, observe.",
    },
    default="I have no canned answer for that — a real model would improvise here.",
)

print(hello_llm.complete(system="You are concise.", user="Say hello to the bootcamp"))

## 2. Exercise: ask three questions

**Context.** `hello_llm` knows two keywords, `agent` and `hello`. Anything else gets the default. Three calls show all three paths.

**Instructions.**

1. Case 1 is done: a question containing `agent`.
2. Add case 2: a question containing `hello`.
3. Add case 3: a question that matches neither keyword.
4. Run the cell, read the three answers, then run the check.

In [ ]:
answers = []
answers.append(hello_llm.complete(system="You are concise.", user="What is an agent?"))  # case 1, done
# TODO(you): case 2, a question containing "hello"
# TODO(you): case 3, a question matching neither keyword

for reply in answers:
    print(reply)

**Expected output** (yours may differ in wording, not in shape):

```
An agent is a loop around a model: perceive, decide, act, observe.
Hello! I am a deterministic stand-in for a language model.
I have no canned answer for that — a real model would improvise here.
✅ ch01-e1 passed
```

In [ ]:
check("ch01-e1", answers)

## 3. Where does the model actually run?

Inspect the seam: `FakeLLM`, `OllamaClient`, `AnthropicClient`, and `OpenAICompatibleClient` all satisfy the same one-method protocol. Swapping providers changes **no** application code. `LIVE`, from the preflight cell, is whichever lane your `.env` picked.

In [ ]:
import inspect

from bootcamp_agent import llm

print(inspect.getsource(llm.LLMClient))
print(f"LIVE is a {type(LIVE).__name__}")

## 4. Exercise: the same question, two lanes

**Context.** The fake is *deterministic*: same question, same answer, every time. A real model is not. That difference is why this course spends a whole week on evaluation.

**Instructions.**

1. The `fake` pair is done: the same question twice through `hello_llm`.
2. Fill the `live` pair: the same question twice through `LIVE`.
3. Run the cell. Compare the two pairs. Then run the check.
4. On the ollama lane the two live answers may differ in wording. On the fake lane they are identical. Both are correct.

In [ ]:
question = "Explain in one sentence what an agent is."

runs = {
    "fake": [hello_llm.complete(system="You are concise.", user=question) for _ in range(2)],  # done
    "live": [],  # TODO(you): the same question, twice, through LIVE
}

for lane, pair in runs.items():
    print(f"[{lane}] same answer twice? {pair[0] == pair[1] if len(pair) == 2 else 'not run yet'}")
    for reply in pair:
        print(f"   {reply[:100]}")

**Expected output** (yours may differ in wording, not in shape):

```
[fake] same answer twice? True
   An agent is a loop around a model: perceive, decide, act, observe.
   An agent is a loop around a model: perceive, decide, act, observe.
[live] same answer twice? False        <- True on the fake lane, usually False on ollama
   An agent is a program that uses a model to decide which actions to take toward a goal.
   An agent is software that plans, calls tools, and checks results until a task is done.
✅ ch01-e2 passed
```

In [ ]:
check("ch01-e2", runs)

## 5. Exercise: your reliability bar

**Context.** Reliability is a decision you make before the first real answer, not after. Write yours down so the evaluation week has a target.

**Instructions.**

1. The first sentence is done as an example. Replace it with your own.
2. Write the other two sentences. Be concrete: name a kind of task, a kind of data, a kind of risk.
3. Run the check. It only measures that each sentence is yours and complete.

In [ ]:
reliability_bar = {
    "reliable_when": "it cites a corpus document I can open, or plainly says it does not know.",  # example, replace
    "review_when": "...",  # TODO(you): I will always require human review when ...
    "never_unreviewed": "...",  # TODO(you): one task I will NOT let an assistant do unreviewed
}
for key, sentence in reliability_bar.items():
    print(f"{key:18} {sentence}")

**Expected output** (yours may differ in wording, not in shape):

```
reliable_when      it cites a corpus document I can open, or plainly says it does not know.
review_when        the answer would change money, credentials, or anything in production.
never_unreviewed   rotate a secret or delete a branch on the shared repository.
✅ ch01-e3 passed
```

In [ ]:
check("ch01-e3", reliability_bar)

## Exit ticket

- What works? What is unclear?
- Homework: finish `SETUP.md`, screenshot the doctor, and write three developer tasks an assistant may help with but must not complete without review.

## Review

The scorecard for this notebook. Every ❌ line names the exercise and the hint.

In [ ]:
review("ch01")